<a href="https://colab.research.google.com/github/DFuc1302/CCTV_keyframes_retrieving/blob/main/notebooks/CCTV_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CCTV: Colab batch → Windows UI

## Goal
Xử lý job xuất từ UI bằng GPU Colab, lưu checkpoint vào Drive, tải result ZIP về UI Windows.
**Validation: demo GPU→Windows UI đã đạt trên Tesla T4/Python 3.11.13, 12 stages và 24 frames của video test-pattern 12 giây.** Smoke dùng source archive exact commit vì GitHub clone thiếu authentication; xem [evidence và giới hạn](https://github.com/DFuc1302/CCTV_keyframes_retrieving/blob/main/docs/CCTV_FULLVIDEO_COLAB.md). Bạn cần chạy cells với source access/quota của mình.
Mọi dữ liệu là demo; M8 pending. 2 FPS có thể bỏ lỡ chuyển động nhanh và không thay thế kiểm định chất lượng.

## Setup
Trong UI Windows: chọn video → camera/ROI → toàn video hoặc khoảng → lưu → **Xuất job Colab**.
Trong Colab: Runtime → Change runtime type → GPU. Kiểm tra quota tài khoản; không mua thêm compute units.
Google AI Pro hiện có quyền lợi Colab tùy điều kiện tài khoản: [Google One](https://support.google.com/googleone/answer/14534406?hl=en).
GPU/runtime không được đảm bảo: [Colab FAQ](https://research.google.com/colaboratory/faq.html).


### 1. Đăng nhập, cấp quyền Drive và chọn job ZIP
Dùng đúng tài khoản có quota. Đặt ZIP vào thư mục `MyDrive/CCTV_Colab`; sửa tên file bên dưới. Drive chỉ lưu input/checkpoint/model cache/result, xử lý trên ổ local Colab.

In [12]:
from google.colab import drive
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys
import zipfile
import time

drive.mount('/content/drive')
persistent = Path('/content/drive/MyDrive/CCTV_Colab')
persistent.mkdir(parents=True, exist_ok=True)
job_name = 'colab-job.zip'  # Đổi thành tên ZIP bạn đã xuất từ UI.
drive_job = persistent / job_name
if not drive_job.is_file():
    raise FileNotFoundError(f'Upload job ZIP to {drive_job} using Google Drive first')
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('This dependency profile supports Python 3.10–3.12; select a compatible Colab runtime')
job = Path('/content/cctv-job.zip')
shutil.copyfile(drive_job, job)
with zipfile.ZipFile(job) as package:
    info = package.getinfo('cloud.json')
    if info.file_size > 16 * 1024**2:
        raise ValueError('Oversized cloud metadata')
    metadata = json.loads(package.read('cloud.json'))
commit = metadata['commit']
if metadata['kind'] != 'cctv-cloud-job-v1' or not re.fullmatch('[0-9a-f]{40}', commit):
    raise ValueError('Invalid exported job/commit')
print('Job commit:', commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Job commit: 791c85a1e72e704fcefbfbd3b6883da4203bf54e


### 2. Checkout đúng phiên bản và môi trường riêng
Dùng lại torch/torchvision GPU có sẵn trong Colab. Packages bổ sung và pip cache nằm trong môi trường project/Drive. Nếu NVIDIA GPU không khả dụng, dừng và dùng UI local.

In [ ]:
import os
repo = Path('/content/CCTV_keyframes_retrieving')
repository_url = 'https://github.com/DFuc1302/CCTV_keyframes_retrieving.git'
if not repo.exists():
    subprocess.run(['git', 'clone', repository_url, str(repo)], check=True)
subprocess.run(['git', 'fetch', 'origin', commit], cwd=repo, check=True)
subprocess.run(['git', 'checkout', '--detach', commit], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(repo / '.venv-colab')], check=True)
python = repo / '.venv-colab/bin/python'
environment = dict(os.environ, PIP_CACHE_DIR=str(persistent / 'pip-cache'),
                   YOLO_CONFIG_DIR=str(repo / 'outputs/model-cache/ultralytics'))
subprocess.run([str(python), '-m', 'pip', 'install', '-r', str(repo / 'requirements/colab-runtime.txt')],
               cwd=repo, env=environment, check=True)
# Reuse Colab's working CUDA stack; pip may have added shadow copies to the venv.
cuda_query = "import importlib.metadata as m,json,sys; from pathlib import Path; print(json.dumps([d.metadata['Name'] for d in m.distributions() if d.metadata['Name'].lower().startswith('nvidia-') and Path(d.locate_file('')).is_relative_to(Path(sys.prefix))]))"
venv_cuda = json.loads(subprocess.check_output([str(python), '-c', cuda_query], cwd=repo, env=environment, text=True))
if venv_cuda:
    subprocess.run([str(python), '-m', 'pip', 'uninstall', '-y', *venv_cuda], cwd=repo, env=environment, check=True)
subprocess.run([str(python), 'scripts/verify_runtime.py', '--require-cuda'], cwd=repo, env=environment, check=True)
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    raise RuntimeError('This Colab runtime lacks ffmpeg/ffprobe; use a standard supported runtime')


### 3. Tải hoặc khôi phục model cache
CLIP và YOLO dùng đúng revisions/checksum của pipeline local. Cache được sao chép trước khi xử lý; mỗi subprocess chỉ giữ model của một bước trong VRAM.

In [ ]:
from contextlib import redirect_stdout
cache = repo / 'outputs/model-cache'
saved_cache = persistent / 'model-cache'
if saved_cache.exists() and not cache.exists():
    shutil.copytree(saved_cache, cache)
for module in ('scripts.download_cctv_clip', 'scripts.download_cctv_yolo'):
    subprocess.run([str(python), '-m', module], cwd=repo, env=environment, check=True)
if not saved_cache.exists():
    shutil.copytree(cache, saved_cache, ignore=shutil.ignore_patterns('.ingest.lock'))


## Steps
### 4. Chạy batch và resume
Pipeline kiểm tra toàn ZIP/checksum/code, dùng lại các bước đã lưu và segment tracking đã commit.
Nếu runtime ngắt: kết nối lại GPU, chạy lại các cell với cùng ZIP, cùng phiên bản môi trường và thư mục Drive.
Nếu môi trường/checksum đổi, pipeline sẽ từ chối resume; dùng job mới thay vì sửa checkpoint.


In [ ]:
started = time.perf_counter()
runner = "from pathlib import Path; import sys; from pipeline.cctv.cloud import run_job; print(run_job(Path.cwd(), Path(sys.argv[1]), Path(sys.argv[2])))"
subprocess.run([str(python), '-c', runner, str(job), str(persistent)], cwd=repo, env=environment, check=True)
import hashlib
checksum = hashlib.sha256()
with job.open('rb') as handle:
    for block in iter(lambda: handle.read(1024**2), b''):
        checksum.update(block)
result = persistent / (checksum.hexdigest() + '.result.zip')
if not result.is_file():
    raise RuntimeError('Result was not published; inspect run-colab/console.log')
print('Result:', result)
print('Batch wall seconds including result ZIP:', round(time.perf_counter() - started, 3))


## Checks
### 5. Kiểm tra timing và tải ZIP
Timing riêng từng stage nằm trong ZIP; thời gian batch gồm checkpoint/ZIP, không gồm upload hoặc cài môi trường/tải model.
Không dùng số local để tự suy ra tốc độ Colab. Lưu log GPU/package versions và đo cùng video/cấu hình trước khi so sánh.


In [ ]:
from google.colab import files
with zipfile.ZipFile(result) as package:
    for name in ('run/cloud-environment.json', 'run/cloud-runtime.json'):
        print(name, package.read(name).decode())
    print(package.read('run/stage-timings.jsonl').decode()[-8000:])
files.download(str(result))


## Next Steps
Trong UI Windows, chọn **đúng bộ dữ liệu và camera/ROI đã xuất** → **Nhập kết quả Colab** → chọn result ZIP.
UI kiểm tra/checksum và kích hoạt run mới; sau đó dùng tìm kiếm, traffic, overlay và gợi ý mô phỏng như local.
Giới hạn ZIP 4 GiB, tối đa giải nén 20 GiB; bắt đầu với 1 video demo ngắn.
Khi tải xong, Runtime → Disconnect and delete runtime để giải phóng GPU.
Đăng nhập, quota và mount Drive cần bạn thao tác. Demo GPU→UI và benchmark tracking sáu MOV ở cả 2 FPS/native đã được đo; xem docs/CCTV_FULLVIDEO_COLAB.md để biết timing và giới hạn. M8, review độc lập và máy thứ hai vẫn chưa nghiệm thu.
